# City population Markov Cahain
This is a simple implementation of a Markov Chain to model city populations over time. The model assumes that the population of a city can change based on certain probabilities.
A total of 20 cities are modeled, the population of each city can increase or decrease based on defined probabilities over iterations.
```python

In [53]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import kagglehub
import folium
import os

dataset_path = kagglehub.dataset_download("justinboon/municipalities-of-the-netherlands")
print(f"Dataset downloaded to: {dataset_path}")

Dataset downloaded to: /Users/jadenvanrijswijk/.cache/kagglehub/datasets/justinboon/municipalities-of-the-netherlands/versions/7


In [54]:
df = pd.read_csv(os.path.join(dataset_path, 'municipalities_v7.csv'))
df = df[['municipality', 'province', 'population', 'surface_km2', 'latitude', 'longitude']]

RANDOM_SEED = 42
CITY_SAMPLE_SIZE = 10
MAX_CITY_CONNECTIONS = 7

print(df.size)
print(df.info())

41040
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6840 entries, 0 to 6839
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   municipality  6840 non-null   object 
 1   province      6840 non-null   object 
 2   population    376 non-null    float64
 3   surface_km2   6804 non-null   float64
 4   latitude      6840 non-null   float64
 5   longitude     6840 non-null   float64
dtypes: float64(4), object(2)
memory usage: 320.8+ KB
None


In [55]:
df = df.dropna()
df = df[df['province'] == 'Noord-Holland']
df = df[df['population'] > 0]
df = df.rename(columns={'municipality': 'city_name'}) 

# get top 20 highest population cities
df = df.nlargest(CITY_SAMPLE_SIZE, 'population').copy().reset_index(drop=True)
starting_city = df[df['city_name'] == 'Amstelveen'].iloc[0]

df

,city_name,province,population,surface_km2,latitude,longitude
0,Amsterdam,Noord-Holland,853312.0,219.3,52.370216,4.895168
1,Haarlem,Noord-Holland,155205.0,32.1,52.387388,4.646219
2,Zaanstad,Noord-Holland,150911.0,83.2,52.457966,4.751043
3,Haarlemmermeer,Noord-Holland,144166.0,185.3,52.300378,4.674359
4,Alkmaar,Noord-Holland,94906.0,31.2,52.632842,4.755037
5,Hilversum,Noord-Holland,86574.0,46.4,52.229170,5.166897
6,Amstelveen,Noord-Holland,85135.0,44.1,52.311421,4.870087
7,Purmerend,Noord-Holland,79552.0,24.6,52.514381,4.964061
8,Hoorn,Noord-Holland,71741.0,53.3,52.642365,5.060212
9,Velsen,Noord-Holland,67231.0,63.1,52.452059,4.630587


In [56]:
def latitude_longtitude_to_distance_km(lat1, lon1, lat2, lon2):
    """ Haversine formula to calculate distance between two lat/lon points in km.
        This function was made with AI assistance.
    """
    R = 6371  # Radius of the Earth in km
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a = np.sin(dlat / 2) ** 2 + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlon / 2) ** 2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    distance = R * c
    return distance

def log_scale_normalize(arr, x):
    """ Log scale normalize an array value to 0-1 range.
        This is used to reduce the impact of large outliers.
    """
    top_log = np.log(x + 1)
    bottom_log = np.log(np.max(arr) + 1)
    return 1 - (top_log / bottom_log)

distance_matrix = np.zeros((len(df), len(df)))
city_names = df['city_name'].tolist()

for i, city1 in df.iterrows():
    for j, city2 in df.iterrows():
        if i != j:
            distance = latitude_longtitude_to_distance_km(city1['latitude'], city1['longitude'], city2['latitude'], city2['longitude'])
            distance_matrix[i, j] = distance

In [ ]:
class MarkovChainNode:
    def __init__(self, city):
        self.city = city
        self.city_index = city_names.index(city['city_name'])
        self.connections = {}
        
        self.transition_weights = {
            'distance': 0.5,
            'population': 0.4,
            'surface_km2': 0.1,
        }
            
        
    def evaluate_attractiveness(self, other_city):
        """ Evaluate the attractiveness of another city based on distance, surface area and population.
            A high population and surface area increases attractiveness, while a high distance decreases it.
            The attractiveness is a value between 0 and 1.
        """
        other_city_index = city_names.index(other_city['city_name'])
        distance_to_other_city = distance_matrix[self.city_index, other_city_index]
        
        distance_factor = distance_matrix[self.city_index] /  distance_to_other_city.max()
        population_factor = df['population'], other_city.population.max()
        surface_area_factor = df['surface_km2'], other_city.surface_km2.max()
        
        attractiveness = [
            self.transition_weights['distance'] * distance_factor,
            self.transition_weights['population'] * population_factor,
            self.transition_weights['surface_km2'] * surface_area_factor,
        ]
        
        print(attractiveness)
        
        return sum(attractiveness)

    def evolve_connections(self):
        """ Establish connections to other cities based on attractiveness.
            Attactiveness is used as a probability to form a connection.
            Connections are bidirectional. Self-connections are allowed.
        """
        attractions = pd.Series(dtype=float)
        for _, other_city in df.iterrows():
            if other_city['city_name'] == self.city['city_name']:
                continue
            
            attractiveness = self.evaluate_attractiveness(other_city)
            attractions.at[other_city['city_name']] = attractiveness
            
        
        connection_probabilities = attractions / attractions.sum() 
        
        self_attrativeness = self.evaluate_attractiveness(self.city)

        n_conns = int((1- self_attrativeness) * MAX_CITY_CONNECTIONS)
        # print(n_conns)
        
        connections = connection_probabilities.sample(n=n_conns, weights=connection_probabilities, replace=False).reset_index()
        
        for conn in connections.itertuples():
            self.connections[conn.index] = conn[0] / MAX_CITY_CONNECTIONS
        
    def evolve_population(self):
        """ Evolve the population of the city based on its connections.
            Each connection has a chance to increase or decrease the population.
            The chance is based on the attractiveness of the connection.
        """
        original_population = self.city['population']
        for conn, moving_percentage in self.connections.items():
            # Move to connected city
            moving_population = int(original_population * moving_percentage * 0.01)
            df.loc[df['city_name'] == self.city['city_name'], 'population'] -= moving_population
            df.loc[df['city_name'] == conn, 'population'] += moving_population
            
    @property
    def name(self):
        return self.city['city_name']
    
    @property
    def population(self):
        return self.city['population']

In [61]:
nodes = [MarkovChainNode(row) for _, row in df.iterrows()]

for node in nodes:
    node.evolve_connections()
    
for node in nodes:
    node.evolve_population()
    
print(df)

[np.float64(0.0875494052764621), np.float64(0.049795799310030156), np.float64(0.03513364959440448)]
[np.float64(0.11541241113376588), np.float64(0.050654206231711824), np.float64(0.017827563856692596)]
[np.float64(0.08844120632655605), np.float64(0.05182726266278142), np.float64(0.003107178480673334)]
[np.float64(0.006822942458843295), np.float64(0.06426498759643398), np.float64(0.035644620486579785)]
[np.float64(0.039381015177606005), np.float64(0.06681134332673407), np.float64(0.02847767934377179)]
[np.float64(0.20771728697139058), np.float64(0.06647738809219939), np.float64(0.02939964546145453)]
[np.float64(0.09001865846096097), np.float64(0.06943530257563739), np.float64(0.03989623337694473)]
[np.float64(0.0), np.float64(0.07246322790758168), np.float64(0.025958638681132486)]
[np.float64(0.06477561561137779), np.float64(0.07426371370870144), np.float64(0.022883190363693684)]
[np.float64(0.5), np.float64(0.0), np.float64(0.0)]
[np.float64(0.11042988828878297), np.float64(0.0), np.fl

In [59]:
m = folium.Map(location=[52.370216, 4.895168], zoom_start=9) # starting zoom on Amsterdam

for row in df.itertuples():
    folium.CircleMarker(
        location=[row.latitude, row.longitude],  # Amsterdam lat/lon
        radius=row.population / 20000,               # population scaling
        popup=row.city_name
    ).add_to(m)
    
connected_cities = set()

for node in nodes:
    for conn, weight in node.connections.items():
        if (node.name, conn) in connected_cities or (conn, node.name) in connected_cities:
            continue  # Skip if this connection has already been drawn
        
        other_city = df[df['city_name'] == conn].iloc[0]
        folium.PolyLine(
            locations=[
                [node.city['latitude'], node.city['longitude']],
                [other_city['latitude'], other_city['longitude']]
            ],
            weight=weight * 5,
            color='blue',
            opacity=0.5
        ).add_to(m)
        connected_cities.add((node.name, conn))

m